In [1]:
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv("movie_long_with_emotions_full.csv")

# 2. 날짜 변환
df["개봉일"] = pd.to_datetime(df["개봉일"], errors="coerce")
df["일자"] = pd.to_datetime(df["일자"], errors="coerce")

# 3. 분석 기간 설정 (2023-01-01 ~ 2024-12-31)
start_date = "2023-01-01"
end_date = "2024-12-31"
df = df[(df["일자"] >= start_date) & (df["일자"] <= end_date)]

# 4. 분석 대상 영화 = 개봉일이 2023~2024인 영화만
df = df[(df["개봉일"] >= start_date) & (df["개봉일"] <= end_date)]

# 5. 영화별 OTT 공개일 계산 (post_netflix == 1인 첫날)
ott_release = (
    df[df["post_netflix"] == 1]
    .groupby("영화명")["일자"]
    .min()
    .reset_index()
    .rename(columns={"일자": "OTT공개일"})
)

# 6. merge
df = df.merge(ott_release, on="영화명", how="left")

# 7. treatment 변수 생성
df["treatment"] = (
    (df["OTT공개일"].notna()) & (df["일자"] >= df["OTT공개일"])
).astype(int)

# 8. outcome 후보 (감정 확률 컬럼들)
prob_cols = [
 '불평/불만_prob','환영/호의_prob','감동/감탄_prob','지긋지긋_prob','고마움_prob',
 '슬픔_prob','화남/분노_prob','존경_prob','기대감_prob','우쭐댐/무시함_prob',
 '안타까움/실망_prob','비장함_prob','의심/불신_prob','뿌듯함_prob','편안/쾌적_prob',
 '신기함/관심_prob','아껴주는_prob','부끄러움_prob','공포/무서움_prob','절망_prob',
 '한심함_prob','역겨움/징그러움_prob','짜증_prob','어이없음_prob','없음_prob',
 '패배/자기혐오_prob','귀찮음_prob','힘듦/지침_prob','즐거움/신남_prob','깨달음_prob',
 '죄책감_prob','증오/혐오_prob','흐뭇함(귀여움/예쁨)_prob','당황/난처_prob','경악_prob',
 '부담/안_내킴_prob','서러움_prob','재미없음_prob','불쌍함/연민_prob','놀람_prob',
 '행복_prob','불안/걱정_prob','기쁨_prob','안심/신뢰_prob'
]

# 9. 최종 패널 데이터 구성
final_cols = ["영화명", "일자", "개봉일", "OTT공개일", "post_netflix", "treatment"] + prob_cols
df_final = df[final_cols]

# 10. 저장
df_final.to_csv("movie_long_staggered_did_emotions.csv", index=False, encoding="utf-8-sig")
print("✅ 전처리 완료 → movie_long_staggered_did_emotions.csv 저장")


✅ 전처리 완료 → movie_long_staggered_did_emotions.csv 저장
